In [1]:
DF_PATH       = "../data/processed/2_cleaned_data.pkl"

ROLE_COLS      = ['DevType']
TECH_COLS      = ['LanguageWorkedWith',
                  'DatabaseWorkedWith',
                  'WebframeWorkedWith',
                  'MiscTechWorkedWith']


MLFLOW_TRACKING_URI = '../models/mlruns'
MLFLOW_EXPERIMENT_NAME = "skills_jobs_stackoverflow"

LOG_PATH = "../models/temp/"
LOG_DATA_PKL    =  "data.pkl"
LOG_MODEL_PKL   =  "model.pkl"
LOG_METRICS_PKL =  "metrics.pkl"

In [2]:
# Load packages
import pandas as pd 
import numpy as np
import logging
import pickle
import random
import plotly 
import os
from pathlib import Path

import mlflow
from mlflow.tracking import MlflowClient

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline, FeatureUnion
from sklearn.feature_selection import VarianceThreshold
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV

from sklearn import metrics
from sklearn.metrics import auc, accuracy_score, confusion_matrix, f1_score, precision_score, recall_score

from sklearn.decomposition import PCA, KernelPCA

from sklearn import tree
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from matplotlib import pyplot as plt

## Functions

In [3]:
def calculate_quality(ground_truth, predictions, metric_function, sort_values=False):
    quality_scores = {}
    for col in predictions.columns:
        role_pred  = predictions[col].copy()
        role_truth = ground_truth[col].copy()
        quality_scores[col] = round(metric_function(role_truth, role_pred) * 100, 2)
        
    quality_scores = pd.Series(quality_scores.values(), index=quality_scores.keys())
    if sort_values:
        quality_scores = quality_scores.sort_values()
    
    return quality_scores

## Prep data

### 1. read data

In [4]:
df = pd.read_pickle(DF_PATH)

### 2.balance classes 

In [5]:
roles_df = df["DevType"].copy()
roles_df.sum()

Academic researcher                              1039
Data or business analyst                         1059
Data scientist or machine learning specialist    1275
Database administrator                            745
DevOps specialist                                1212
Developer, QA or test                             791
Developer, back-end                              9144
Developer, desktop or enterprise applications    2985
Developer, embedded applications or devices      1193
Developer, front-end                             5177
Developer, full-stack                            8718
Developer, game or graphics                       608
Developer, mobile                                2573
Engineer, data                                    916
Scientist                                         590
System administrator                              880
dtype: int64

In [6]:
sample_per_class = 700
resampled_roles = []
for role in roles_df.columns:
    sub_df  = roles_df.loc[roles_df[role]==1].copy()
    if len(sub_df) < sample_per_class:
        sub_df = sub_df.sample(sample_per_class, replace=True, random_state = 0)
    else:
        sub_df = sub_df.sample(sample_per_class, replace=True, random_state = 0)
    resampled_roles.append(sub_df)

In [7]:
## construct dfs
roles_df = pd.concat(resampled_roles)
df = df.loc[roles_df.index].copy()

In [8]:
roles_df.sum()

Academic researcher                              1468
Data or business analyst                         1281
Data scientist or machine learning specialist    1568
Database administrator                           1086
DevOps specialist                                1180
Developer, QA or test                             989
Developer, back-end                              3890
Developer, desktop or enterprise applications    1798
Developer, embedded applications or devices      1065
Developer, front-end                             1836
Developer, full-stack                            2894
Developer, game or graphics                       895
Developer, mobile                                1426
Engineer, data                                   1213
Scientist                                        1136
System administrator                             1157
dtype: int64

## Split to train and test

In [9]:
X_train, X_test, y_train, y_test = train_test_split(df.drop('DevType', axis=1),
                                                    df['DevType'],
                                                    random_state=0)

C:\Users\user\AppData\Local\Temp\ipykernel_13592\4116850138.py:1: PerformanceWarning: dropping on a non-lexsorted multi-index without a level parameter may impact performance.
  X_train, X_test, y_train, y_test = train_test_split(df.drop('DevType', axis=1),


## Train models

### Initialize MLflow

In [10]:
#initialize and experiment
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
client = MlflowClient()
mlflow.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
exp = client.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
if exp is None:
    experiment_id = client.create_experiment(MLFLOW_EXPERIMENT_NAME)
else:
    experiment_id = exp.experiment_id

C:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:177: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance.
  return FileStore(store_uri, store_uri)


### 1. Vanilla forest

In [11]:
rf_clf = make_pipeline(RobustScaler(),
                       PCA(n_components=0.95),
                       ##Keep the minimum number of components that explain 95% of the variance in the data.
                   RandomForestClassifier(n_jobs=8,
                                              verbose=1,
                                              random_state=0))
rf_clf.fit(X_train.values, y_train.values)


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    4.6s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:   11.5s finished


,steps,"[('robustscaler', ...), ('pca', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,with_centering,True
,with_scaling,True
,quantile_range,"(25.0, ...)"
,copy,True
,unit_variance,False
,n_components,0.95
,copy,True


In [12]:
# Evaluate on train set
predictions = pd.DataFrame(rf_clf.predict(X_train.values), columns = y_train.columns)
train_scores = {score.__name__: calculate_quality(y_train,predictions,score) for score in [accuracy_score, f1_score, precision_score, recall_score]}
train_scores = pd.concat(train_scores, axis=1)

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.1s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.5s finished


In [13]:
# Evaluate on test set
predictions =  pd.DataFrame(rf_clf.predict(X_test.values),
                            columns=y_test.columns)
test_scores = {score.__name__: calculate_quality(y_test, predictions, score) 
                for score in [accuracy_score, f1_score, precision_score, recall_score]}
test_scores = pd.concat(test_scores,axis=1)
mean_test_scores = test_scores.mean()

[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.2s finished


In [14]:
print(mean_test_scores)
test_scores.sort_values("precision_score")

accuracy_score     93.699375
f1_score           73.537500
precision_score    90.983125
recall_score       62.121250
dtype: float64


,accuracy_score,f1_score,precision_score,recall_score
"Developer, back-end",82.36,71.58,80.67,64.32
"Developer, full-stack",86.68,70.23,84.94,59.86
"Developer, embedded applications or devices",95.04,68.76,85.96,57.30
"Developer, desktop or enterprise applications",90.39,59.55,88.39,44.90
"Developer, front-end",91.29,66.76,89.09,53.38
Academic researcher,94.79,78.14,90.31,68.87
Database administrator,95.75,73.38,91.11,61.42
Data scientist or machine learning specialist,96.00,84.23,92.00,77.66
Data or business analyst,95.96,77.71,92.49,67.01
"Developer, QA or test",95.11,64.96,92.70,50.00


### Log

In [15]:
## Log
# Data details
data_details = {"data_path": DF_PATH,
                "training_indices": X_train.index.tolist(),
                "test_indices": X_test.index.tolist(),
                "features_names": X_train.columns.droplevel(0).tolist(),
                "targets_names": y_train.columns.tolist()}

with open(os.path.join(LOG_PATH, LOG_DATA_PKL), "wb") as output_file:
    pickle.dump(data_details, output_file)

In [16]:
# Model
model = {"model_description": "Random Forest: with PCA - Basic",
         "model_details": str(rf_clf),
         "model_object": rf_clf}

with open(os.path.join(LOG_PATH, LOG_MODEL_PKL), "wb") as output_file:
    pickle.dump(model, output_file)

In [17]:
# Performance details
classes_metrics = {"train_scores": train_scores,
                   "test_scores": test_scores}

with open(os.path.join(LOG_PATH, LOG_METRICS_PKL), "wb") as output_file:
    pickle.dump(classes_metrics, output_file)

### Start a new run and track

In [18]:
# Start a new run and track
with mlflow.start_run(experiment_id=exp.experiment_id,
                      run_name=model["model_description"]):
    # Log pickles
    mlflow.log_artifacts(LOG_PATH)

    # Track metrics
    for metric, score in mean_test_scores.items():
        mlflow.log_metric(metric, score)

### 2. Random Forest with Non-linearity

In [19]:
rf_clf = make_pipeline(StandardScaler(),
                      FeatureUnion([('linear_pca', PCA(n_components=40)),
                                   ('kernel_pca', KernelPCA(n_components=40, kernel = 'rbf'))]),
                       RandomForestClassifier(random_state=0))
rf_clf.fit(X_train, y_train)

,steps,"[('standardscaler', ...), ('featureunion', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,transformer_list,"[('linear_pca', ...), ('kernel_pca', ...)]"
,n_jobs,None
,transformer_weights,None
,verbose,False


In [20]:
# Evaluate on train set
predictions = pd.DataFrame(rf_clf.predict(X_train.values), columns = y_train.columns)
train_scores = {score.__name__: calculate_quality(y_train,predictions,score) for score in [accuracy_score, f1_score, precision_score, recall_score]}
train_scores = pd.concat(train_scores, axis=1)

In [21]:
# Evaluate on test set
predictions =  pd.DataFrame(rf_clf.predict(X_test.values),
                            columns=y_test.columns)
test_scores = {score.__name__: calculate_quality(y_test, predictions, score) 
                for score in [accuracy_score, f1_score, precision_score, recall_score]}
test_scores = pd.concat(test_scores,axis=1)
mean_test_scores = test_scores.mean()

In [22]:
print(mean_test_scores)
test_scores.sort_values("precision_score")

accuracy_score     93.795000
f1_score           74.293125
precision_score    89.803750
recall_score       63.884375
dtype: float64


,accuracy_score,f1_score,precision_score,recall_score
"Developer, back-end",83.07,73.31,80.47,67.32
"Developer, embedded applications or devices",94.79,67.98,82.01,58.05
"Developer, full-stack",86.68,71.11,82.55,62.45
"Developer, front-end",91.14,66.58,87.28,53.81
"Developer, desktop or enterprise applications",90.36,59.21,88.69,44.44
Data scientist or machine learning specialist,95.86,83.93,89.91,78.70
Data or business analyst,96.11,79.16,90.39,70.41
Academic researcher,94.86,78.38,90.94,68.87
Database administrator,95.75,73.38,91.11,61.42
DevOps specialist,95.07,72.06,91.75,59.33


In [23]:
## Log
# Data details
data_details = {"data_path": DF_PATH,
                "training_indices": X_train.index.tolist(),
                "test_indices": X_test.index.tolist(),
                "features_names": X_train.columns.droplevel(0).tolist(),
                "targets_names": y_train.columns.tolist()}

with open(os.path.join(LOG_PATH, LOG_DATA_PKL), "wb") as output_file:
    pickle.dump(data_details, output_file)

In [24]:
# Model
model = {"model_description": "Random Forest: with PCA - KernelPca",
         "model_details": str(rf_clf),
         "model_object": rf_clf}

with open(os.path.join(LOG_PATH, LOG_MODEL_PKL), "wb") as output_file:
    pickle.dump(model, output_file)

In [25]:
# Performance details
classes_metrics = {"train_scores": train_scores,
                   "test_scores": test_scores}

with open(os.path.join(LOG_PATH, LOG_METRICS_PKL), "wb") as output_file:
    pickle.dump(classes_metrics, output_file)

### start a New run and track

In [26]:
# Start a new run and track
with mlflow.start_run(experiment_id=exp.experiment_id,
                      run_name=model["model_description"]):
    # Log pickles
    mlflow.log_artifacts(LOG_PATH)

    # Track metrics
    for metric, score in mean_test_scores.items():
        mlflow.log_metric(metric, score)